In [1]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision.transforms.functional import resize
from torchvision.transforms import InterpolationMode
import random
import time

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
os.chdir('/home/ntdung/Medical')

In [3]:
os.chdir('/media/sslab/PACS/sslab/nguyentiendung')

In [4]:
excel_path = 'data/participants_1.xlsx'
df = pd.read_excel(excel_path, nrows=None)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44.2,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39.3,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42.5,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,19.7,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20.0,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4919,4944,66.0,control,f,sub-BrainAge023209,RocklandSample
4920,4945,69.0,control,m,sub-BrainAge023210,RocklandSample
4921,4946,23.0,control,m,sub-BrainAge023211,RocklandSample
4922,4947,54.0,control,f,sub-BrainAge023212,RocklandSample


In [5]:
def is_integer(n):
    return float(n).is_integer()

n_total = len(df)
n_integer = df['subject_age'].apply(is_integer).sum()
n_decimal = n_total - n_integer

print(f"Samples with integer age values: {n_integer}")
print(f"Samples with decimal age values: {n_decimal}")

Samples with integer age values: 4218
Samples with decimal age values: 706


In [6]:
df['subject_age'] = df['subject_age'].round().astype(int)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,20,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4919,4944,66,control,f,sub-BrainAge023209,RocklandSample
4920,4945,69,control,m,sub-BrainAge023210,RocklandSample
4921,4946,23,control,m,sub-BrainAge023211,RocklandSample
4922,4947,54,control,f,sub-BrainAge023212,RocklandSample


# Prepare Input

In [7]:
class BrainMRIDataset(Dataset):
    """
    Dataset cho Brain MRI với thông tin tuổi và giới tính thật và phản thực
    """
    def __init__(self, data_dir, participants_file, transform=None, img_size=128):
        self.data_dir = data_dir
        self.transform = transform
        self.img_size = img_size

        self.participants_df = pd.read_excel(participants_file)
        self.participants_df['subject_age'] = self.participants_df['subject_age'].round().astype(int)
        
        # Chuẩn hóa giới tính thành giá trị số
        self.participants_df['gender_code'] = self.participants_df['subject_sex'].map({'m': 0, 'f': 1})
        
        # Chuẩn hóa tuổi (min-max scaling)
        self.min_age = self.participants_df['subject_age'].min()
        self.max_age = self.participants_df['subject_age'].max()
        self.age_range = self.max_age - self.min_age
        
        # Lọc các mẫu có dữ liệu MRI hợp lệ
        valid_subjects = []
        for subject_id in self.participants_df['subject_id']:
            file_path = os.path.join(data_dir, subject_id, 'anat', f"{subject_id}_T1w.nii.gz")
            if os.path.exists(file_path):
                valid_subjects.append(subject_id)
        
        self.participants_df = self.participants_df[self.participants_df['subject_id'].isin(valid_subjects)]
        print(f"Tìm thấy {len(self.participants_df)} mẫu có dữ liệu MRI hợp lệ")
    
    def __len__(self):
        return len(self.participants_df)
    
    def _get_middle_slices(self, subject_id):
        try:
            file_path = os.path.join(self.data_dir, subject_id, 'anat', f"{subject_id}_T1w.nii.gz")
            img = nib.load(file_path)
            img_data = img.get_fdata()
            
            # Lấy 3 lát cắt giữa
            slices = [
                img_data[:, :, img_data.shape[2]//2],  # axial
                img_data[img_data.shape[0]//2, :, :],  # sagittal
                img_data[:, img_data.shape[1]//2, :]   # coronal
            ]
            
            ## Chuyển đổi sang tensor và chuẩn hóa
            processed_slices = []
            for slice_data in slices:
                tensor_slice = torch.from_numpy(slice_data).float()
                # Min-max normalization
                if tensor_slice.max() > tensor_slice.min():  # Tránh chia cho 0
                    tensor_slice = (tensor_slice - tensor_slice.min()) / (tensor_slice.max() - tensor_slice.min())
                    
                # Resize slice về kích thước cố định (128x128)
                tensor_slice = tensor_slice.unsqueeze(0)  # Thêm chiều kênh [1, H, W]
                resized_slice = F.interpolate(
                    tensor_slice.unsqueeze(0),  # Thêm chiều batch [1, 1, H, W]
                    size=(self.img_size, self.img_size),
                    mode='bilinear',
                    align_corners=False
                ).squeeze(0)  # Loại bỏ chiều batch [1, H, W]
                
                if self.transform:
                    resized_slice = self.transform(resized_slice)

                processed_slices.append(resized_slice)
            
            # Ghép các lát cắt thành một tensor [3, H, W]
            return torch.cat(processed_slices, dim=0)
            
        except Exception as e:
            print(f"Lỗi khi xử lý {subject_id}: {e}")
            # Trả về tensor 0 trong trường hợp lỗi
            return torch.zeros((3, 130, 130), dtype=torch.float32)
    
    def normalize_age(self, age):
        """Chuẩn hóa tuổi về khoảng [0, 1]"""
        return (age - self.min_age) / self.age_range if self.age_range > 0 else 0
    
    def __getitem__(self, idx):
        """
        Lấy một mẫu từ dataset
        """
        # Lấy thông tin mẫu thật
        real_info = self.participants_df.iloc[idx]
        real_id = real_info['subject_id']
        real_age = real_info['subject_age']
        real_gender = real_info['gender_code']
        
        # Lấy thông tin phản thực từ một mẫu ngẫu nhiên khác
        valid_indices = [i for i in range(len(self)) if i != idx]
        if not valid_indices:  # Nếu chỉ có 1 mẫu trong dataset
            cf_info = real_info
        else:
            cf_idx = random.choice(valid_indices)
            cf_info = self.participants_df.iloc[cf_idx]
        
        cf_id = cf_info['subject_id']
        cf_age = cf_info['subject_age']
        cf_gender = cf_info['gender_code']

        # Lấy ảnh và chuẩn hóa điều kiện
        real_img = self._get_middle_slices(real_id)
        
        # Chuẩn bị điều kiện
        real_condition = torch.tensor([self.normalize_age(real_age), real_gender], dtype=torch.float32)
        cf_condition = torch.tensor([self.normalize_age(cf_age), cf_gender], dtype=torch.float32)
        
        # Cũng chuẩn bị điều kiện gốc (không chuẩn hóa) cho việc kiểm tra/ghi log
        real_raw_condition = torch.tensor([real_age, real_gender], dtype=torch.float32)
        cf_raw_condition = torch.tensor([cf_age, cf_gender], dtype=torch.float32)
        
        return {
            'real_id': real_id,
            'cf_id': cf_id,
            'real_img': real_img,  # Shape: [3, H, W]
            'real_condition': real_condition,  # Shape: [2] - chuẩn hóa
            'cf_condition': cf_condition,  # Shape: [2] - chuẩn hóa
            'real_raw_condition': real_raw_condition,  # Shape: [2] - không chuẩn hóa
            'cf_raw_condition': cf_raw_condition  # Shape: [2] - không chuẩn hóa
        }

In [8]:
dataset = BrainMRIDataset(data_dir='data', participants_file='data/participants_1.xlsx')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

Tìm thấy 4924 mẫu có dữ liệu MRI hợp lệ


In [9]:
sample = dataset[0]

print("Real ID:", sample['real_id'])
print("Counterfactual ID:", sample['cf_id'])
print("Real image shape:", sample['real_img'].shape)
print("Real condition (normalized):", sample['real_condition'])
print("Counterfactual condition (normalized):", sample['cf_condition'])
print("Real condition (raw):", sample['real_raw_condition'])
print("Counterfactual condition (raw):", sample['cf_raw_condition'])

Real ID: sub-BrainAge000019
Counterfactual ID: sub-BrainAge019169
Real image shape: torch.Size([3, 128, 128])
Real condition (normalized): tensor([0.3291, 0.0000])
Counterfactual condition (normalized): tensor([0.0759, 1.0000])
Real condition (raw): tensor([44.,  0.])
Counterfactual condition (raw): tensor([24.,  1.])


In [10]:
batch = next(iter(dataloader))

print("Real image batch shape:", batch['real_img'].shape)
print("Real condition (normalized):", batch['real_condition'], batch['real_condition'].shape)
print("Counterfactual condition (normalized):", batch['cf_condition'], batch['cf_condition'].shape)
print("Real IDs:", batch['real_id'])
print("Counterfactual IDs:", batch['cf_id'])

Real image batch shape: torch.Size([4, 3, 128, 128])
Real condition (normalized): tensor([[0.4051, 1.0000],
        [0.3924, 0.0000],
        [0.3924, 0.0000],
        [0.0506, 0.0000]]) torch.Size([4, 2])
Counterfactual condition (normalized): tensor([[0.0127, 1.0000],
        [0.6709, 0.0000],
        [0.0253, 0.0000],
        [0.3544, 0.0000]]) torch.Size([4, 2])
Real IDs: ['sub-BrainAge022208', 'sub-BrainAge005325', 'sub-BrainAge006415', 'sub-BrainAge022226']
Counterfactual IDs: ['sub-BrainAge010921', 'sub-BrainAge023109', 'sub-BrainAge005693', 'sub-BrainAge005355']


# Generator

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    """Khối tích chập cơ bản cho U-Net"""
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.conv(x)

class SpatialTransformer(nn.Module):
    """Lớp biến đổi không gian để áp dụng biến dạng vào ảnh"""
    def __init__(self):
        super(SpatialTransformer, self).__init__()
    
    def forward(self, src, flow):
        """
        src: tensor hình ảnh nguồn [B, C, H, W]
        flow: tensor trường vector biến dạng [B, 2, H, W]
        """
        # Tạo lưới tọa độ chuẩn
        B, C, H, W = src.size()
        
        # Tạo lưới tọa độ chuẩn (từ -1 đến 1)
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, H, device=src.device),
            torch.linspace(-1, 1, W, device=src.device),
            indexing='ij'  # Thêm indexing='ij' để tương thích với PyTorch mới
        )
        grid = torch.stack((grid_x, grid_y), dim=2).unsqueeze(0)
        grid = grid.expand(B, H, W, 2)
        
        # Chuyển đổi flow từ pixel sang giá trị chuẩn hóa (-1 đến 1)
        flow_grid = flow.permute(0, 2, 3, 1)  # [B, H, W, 2]
        flow_grid = flow_grid / torch.tensor([W/2, H/2], device=flow.device)
        
        # Cộng flow vào lưới cơ sở để tạo ra lưới biến dạng
        sample_grid = grid + flow_grid
        
        # Áp dụng lưới biến dạng để lấy mẫu từ ảnh gốc
        output = F.grid_sample(src, sample_grid, mode='bilinear', padding_mode='border', align_corners=True)
        
        return output

class ScalingAndSquaring(nn.Module):
    """Lớp scaling and squaring để tích hợp trường vận tốc thành trường biến dạng"""
    def __init__(self, n_steps=7):
        super(ScalingAndSquaring, self).__init__()
        self.n_steps = n_steps
        self.transformer = SpatialTransformer()
        
    def forward(self, velocity):
        """
        velocity: tensor trường vận tốc [B, 2, H, W]
        return: trường biến dạng phi(x) thông qua tích hợp scaling and squaring
        """
        # Chia nhỏ trường vận tốc
        flow = velocity / (2 ** self.n_steps)
        
        # Thực hiện scaling and squaring
        for _ in range(self.n_steps):
            flow = flow + self.transformer(flow, flow)
            
        return flow

class Generator(nn.Module):
    """Generator U-Net với scaling and squaring layers"""
    def __init__(self, in_channels=5, init_features=16):
        """
        in_channels: số kênh đầu vào (3 cho ảnh + 2 cho điều kiện)
        """
        super(Generator, self).__init__()
        
        # Encoder
        self.encoder1 = ConvBlock(in_channels, init_features)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder2 = ConvBlock(init_features, init_features*2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.encoder3 = ConvBlock(init_features*2, init_features*2)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bridge
        self.bridge = ConvBlock(init_features*2, init_features*2)
        
        # Decoder - Sử dụng Upsample thay vì ConvTranspose2d
        self.upconv3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features*2, init_features*2, kernel_size=3, padding=1)
        )
        self.decoder3 = ConvBlock(init_features*4, init_features*2)
        
        self.upconv2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features*2, init_features*2, kernel_size=3, padding=1)
        )
        self.decoder2 = ConvBlock(init_features*4, init_features)
        
        self.upconv1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(init_features, init_features, kernel_size=3, padding=1)
        )
        self.decoder1 = ConvBlock(init_features*2, init_features)
        
        # Velocity field prediction (2 kênh cho x và y)
        self.velocity = nn.Conv2d(init_features, 2, kernel_size=3, padding=1)
        
        # Scaling and squaring layer
        self.scaling_squaring = ScalingAndSquaring(n_steps=7)
        
        # Spatial transformer để áp dụng biến dạng
        self.transformer = SpatialTransformer()
        
    def forward(self, x, condition):
        """
        x: tensor ảnh đầu vào [B, 3, H, W]
        condition: tensor điều kiện mục tiêu [B, 2]
        """
        batch_size, _, H, W = x.size()
        
        # Mở rộng điều kiện thành kênh không gian
        condition = condition.view(batch_size, 2, 1, 1).expand(-1, -1, H, W)
        
        # Ghép ảnh và điều kiện
        x_in = torch.cat([x, condition], dim=1)  # [B, 5, H, W]
        
        # Encoder
        enc1 = self.encoder1(x_in)
        enc1_pool = self.pool1(enc1)
        
        enc2 = self.encoder2(enc1_pool)
        enc2_pool = self.pool2(enc2)
        
        enc3 = self.encoder3(enc2_pool)
        enc3_pool = self.pool3(enc3)
        
        # Bridge
        bridge = self.bridge(enc3_pool)
        
        # Decoder với skip connections
        up3 = self.upconv3(bridge)
        
        # Đảm bảo kích thước khớp nhau
        if up3.shape[2:] != enc3.shape[2:]:
            up3 = F.interpolate(up3, size=enc3.shape[2:], mode='bilinear', align_corners=True)
            
        up3 = torch.cat([up3, enc3], dim=1)
        dec3 = self.decoder3(up3)
        up2 = self.upconv2(dec3)
        
        # Đảm bảo kích thước khớp nhau
        if up2.shape[2:] != enc2.shape[2:]:
            up2 = F.interpolate(up2, size=enc2.shape[2:], mode='bilinear', align_corners=True)
            
        up2 = torch.cat([up2, enc2], dim=1)
        dec2 = self.decoder2(up2)
        up1 = self.upconv1(dec2)
        
        # Đảm bảo kích thước khớp nhau
        if up1.shape[2:] != enc1.shape[2:]:
            up1 = F.interpolate(up1, size=enc1.shape[2:], mode='bilinear', align_corners=True)
            
        up1 = torch.cat([up1, enc1], dim=1)
        dec1 = self.decoder1(up1)
        
        # Dự đoán trường vận tốc
        velocity = self.velocity(dec1)
        
        # Tích hợp trường vận tốc để có trường biến dạng
        flow = self.scaling_squaring(velocity)
        
        # Áp dụng trường biến dạng vào ảnh gốc
        deformed_image = self.transformer(x, flow)
        
        return deformed_image, flow

class MultiSliceGenerator(nn.Module):
    """Mô hình Generator xử lý đồng thời 3 lát cắt MRI"""
    def __init__(self):
        super(MultiSliceGenerator, self).__init__()
        # Mỗi lát cắt được xử lý bởi một generator riêng
        self.generators = nn.ModuleList([
            Generator(in_channels=3+2)  # 3 channels cho ảnh + 2 cho điều kiện
            for _ in range(3)
        ])
        
    def forward(self, x, condition):
        """
        x: tensor ảnh gốc [B, 3, H, W]
        condition: tensor điều kiện mục tiêu [B, 2]
        """
        outputs = []
        flows = []
        
        # Xử lý mỗi lát cắt riêng biệt
        for i in range(3):
            slice_img = x[:, i:i+1, :, :]  # Lấy 1 lát cắt [B, 1, H, W]
            
            # Nhân rộng lát cắt thành 3 kênh để phù hợp với đầu vào của Generator
            slice_img = slice_img.repeat(1, 3, 1, 1)  # [B, 3, H, W]
            
            # Đưa vào generator
            output, flow = self.generators[i](slice_img, condition)
            
            # Lấy kênh đầu tiên của output (vì cả 3 kênh giống nhau)
            output = output[:, 0:1, :, :]  # [B, 1, H, W]
            
            outputs.append(output)
            flows.append(flow)
        
        # Ghép 3 lát cắt đã xử lý
        generated_img = torch.cat(outputs, dim=1)  # [B, 3, H, W]
        
        return generated_img, flows

In [12]:
def test_generator():
    """Hàm kiểm tra Generator với kích thước đầu vào 128x128"""
    # Thiết lập seed cho tính khả lặp lại
    torch.manual_seed(42)
    
    # Tạo đầu vào giả
    batch_size = 2
    img_size = 128
    
    # Tạo ảnh đầu vào giả với 3 slice
    fake_img = torch.rand(batch_size, 3, img_size, img_size)
    
    # Tạo điều kiện giả (tuổi đã chuẩn hóa và giới tính)
    fake_condition = torch.tensor([[0.5, 0], [0.7, 1]], dtype=torch.float32)
    
    # Tạo mô hình
    model = MultiSliceGenerator()
    print("Mô hình MultiSliceGenerator đã được khởi tạo.")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Discriminator total parameters: {total_params:,}")

    # Chuyển sang chế độ evaluation
    model.eval()
    
    # Thực hiện forward pass
    with torch.no_grad():
        output_img, output_flows = model(fake_img, fake_condition)
    
    # In kích thước đầu ra
    print("\nKết quả:")
    print(f"Input image shape: {fake_img.shape}")
    print(f"Output image shape: {output_img.shape}")
    print(f"Number of flow fields: {len(output_flows)}")
    for i, flow in enumerate(output_flows):
        print(f"Flow {i+1} shape: {flow.shape}")
    
    return output_img, output_flows

# Chạy hàm test
if __name__ == "__main__":
    test_generator()

Mô hình MultiSliceGenerator đã được khởi tạo.
Discriminator total parameters: 365,862

Kết quả:
Input image shape: torch.Size([2, 3, 128, 128])
Output image shape: torch.Size([2, 3, 128, 128])
Number of flow fields: 3
Flow 1 shape: torch.Size([2, 2, 128, 128])
Flow 2 shape: torch.Size([2, 2, 128, 128])
Flow 3 shape: torch.Size([2, 2, 128, 128])


# Discriminator

In [13]:
def weights_init_normal(model):
    '''
    Khởi tạo trọng số ổn định hơn cho mô hình so với khởi tạo mặc định của PyTorch
    :param model: mô hình cần khởi tạo
    :return:
    '''
    classname = model.__class__.__name__
    # Chỉ áp dụng khởi tạo cho các lớp Conv2d, không phải các module tổng hợp có "Conv" trong tên
    if classname == "Conv2d":
        torch.nn.init.normal_(model.weight.data, 0.0, 0.02)
        if model.bias is not None:
            torch.nn.init.constant_(model.bias.data, 0.0)
    # Khởi tạo cho các lớp BatchNorm
    elif classname.find("BatchNorm2d") != -1:
        torch.nn.init.normal_(model.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(model.bias.data, 0.0)
    # Khởi tạo cho các lớp Linear
    elif classname.find("Linear") != -1:
        torch.nn.init.normal_(model.weight.data, 0.0, 0.02)
        if model.bias is not None:
            torch.nn.init.constant_(model.bias.data, 0.0)

def conv_layer_2d(in_channel, out_channel, maxpool=True, kernel_size=3, padding=1, maxpool_stride=2):
    '''
    Tạo một block convolutional 2D gồm Conv2D, BatchNorm2D, (MaxPool2D tùy chọn), và ReLU
    '''
    if maxpool is True:
        layer = nn.Sequential(
            nn.Conv2d(in_channel, out_channel, padding=padding, kernel_size=kernel_size),
            nn.BatchNorm2d(out_channel),
            nn.MaxPool2d(2, stride=maxpool_stride),
            nn.ReLU(),
        )
    else:
        layer = nn.Sequential(
            nn.Conv2d(in_channel, out_channel, padding=padding, kernel_size=kernel_size),
            nn.BatchNorm2d(out_channel),
            nn.ReLU()
        )
    return layer

class Discriminator(nn.Module):

    def __init__(self, in_channels=3, channel_number=[16, 32, 64, 128, 256, 64]):
        '''
        Discriminator hoàn toàn convolutional cho GAN sinh ảnh phản thực
        :param in_channels: Số kênh đầu vào (3 lát cắt từ 3 trục)
        :param channel_number: Số kênh cho các lớp convolutional
        '''
        super(Discriminator, self).__init__()
        
        n_layer = len(channel_number)
        self.feature_extractor = nn.Sequential()
        
        # Xây dựng mạng trích xuất đặc trưng
        for i in range(n_layer):
            if i == 0:
                in_channel = in_channels  # Bắt đầu với số kênh đầu vào (3)
            else:
                in_channel = channel_number[i - 1]
            
            out_channel = channel_number[i]
            
            if i < n_layer - 1:
                # Sử dụng maxpool cho tất cả các lớp trừ lớp cuối
                self.feature_extractor.add_module(f'conv_{i}',
                                                  conv_layer_2d(in_channel,
                                                            out_channel,
                                                            maxpool=True,
                                                            kernel_size=3,
                                                            padding=1))
            else:
                # Lớp cuối không dùng maxpool
                self.feature_extractor.add_module(f'conv_{i}',
                                                  conv_layer_2d(in_channel,
                                                            out_channel,
                                                            maxpool=False,
                                                            kernel_size=1,
                                                            padding=0))

        in_channel = channel_number[-1]
        
        # Nhánh phân loại real/fake (adversarial)
        self.classifier_adv = nn.Sequential(
            nn.Conv2d(in_channel, 1, kernel_size=3, padding=0, bias=False),
            nn.AdaptiveAvgPool2d(1),  # Global average pooling để giảm kích thước xuống 1x1
            nn.Flatten(),
            nn.Sigmoid()  # Sigmoid cho phân loại nhị phân (real/fake)
        )
        
        # Nhánh phân loại giới tính
        self.classifier_gender = nn.Sequential(
            nn.Conv2d(in_channel, 8, kernel_size=3, padding=0, bias=False),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # Global average pooling
            nn.Flatten(),
            nn.Linear(8, 1),
            nn.Sigmoid()  # Sigmoid cho phân loại nhị phân (nam/nữ)
        )
        
        # Nhánh hồi quy tuổi
        self.classifier_age = nn.Sequential(
            nn.Conv2d(in_channel, 16, kernel_size=3, padding=0, bias=False),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # Global average pooling
            nn.Flatten(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)  # Không có activation để dự đoán giá trị thực
        )

    def forward(self, x):
        '''
        :param x: Tensor hình ảnh đầu vào có dạng (batch_size, 3, H, W)
        :return: Tuple gồm (adv_preds, gender_preds, age_preds) tương ứng với 3 đầu ra
        '''
        # Trích xuất đặc trưng chung cho cả 3 tác vụ
        encoded_features = self.feature_extractor(x)
        
        # Áp dụng 3 nhánh dự đoán
        adv_preds = self.classifier_adv(encoded_features)
        gender_preds = self.classifier_gender(encoded_features)
        age_preds = self.classifier_age(encoded_features)
        
        return adv_preds, gender_preds, age_preds

In [14]:
# Hàm kiểm tra kích thước đầu ra
def test_discriminator():
    batch_size = 4
    height, width = 128, 128  # Giả sử kích thước ảnh đầu vào là 128x128
    
    # Tạo tensor đầu vào ngẫu nhiên
    x = torch.randn(batch_size, 3, height, width)
    
    # Khởi tạo discriminator
    discriminator = Discriminator()
    
    total_params = sum(p.numel() for p in discriminator.parameters())
    print(f"Discriminator total parameters: {total_params:,}")
          
    # Áp dụng discriminator
    adv_preds, gender_preds, age_preds = discriminator(x)
    
    # In kích thước đầu ra
    print(f"Discriminator input shape: {x.shape}")
    print(f"Adversarial predictions shape: {adv_preds.shape}")  # Kỳ vọng: [batch_size, 1]
    print(f"Gender predictions shape: {gender_preds.shape}")    # Kỳ vọng: [batch_size, 1]
    print(f"Age predictions shape: {age_preds.shape}")          # Kỳ vọng: [batch_size, 1]
    
    return discriminator

if __name__ == "__main__":
    test_discriminator()

Discriminator total parameters: 424,730
Discriminator input shape: torch.Size([4, 3, 128, 128])
Adversarial predictions shape: torch.Size([4, 1])
Gender predictions shape: torch.Size([4, 1])
Age predictions shape: torch.Size([4, 1])


# Training

In [ ]:
from tqdm import tqdm

# Định nghĩa các hàm loss
def adversarial_loss(preds, target):
    """Loss cho phân loại real/fake"""
    return nn.BCELoss()(preds, target)

def gender_loss(preds, target):
    """Loss cho phân loại giới tính"""
    return nn.BCELoss()(preds, target)

def age_loss(preds, target):
    """Loss cho hồi quy tuổi"""
    return nn.L1Loss()(preds, target)

def train_gan(generator, discriminator, dataloader, num_epochs=100, 
              lr_g=2e-4, lr_d=2e-4, beta1=0.5, beta2=0.999,
              lambda_adv=1.0, lambda_gender=1.0, lambda_age=1.0,
              checkpoint_dir='checkpoints', save_freq=5, device=None,
              result_dir='results', img_dir='images'):
    """
    Training loop cho GAN MRI phản thực
    
    Args:
        generator: Mô hình Generator
        discriminator: Mô hình Discriminator
        dataloader: DataLoader cho dữ liệu training
        num_epochs: Số epoch training
        lr_g: Learning rate cho Generator
        lr_d: Learning rate cho Discriminator
        beta1, beta2: Các tham số cho Adam optimizer
        lambda_adv, lambda_gender, lambda_age: Trọng số cho các loss
        checkpoint_dir: Thư mục lưu checkpoint
        save_freq: Tần suất lưu checkpoint (số epoch)
        device: Thiết bị tính toán (CPU/GPU)
        result_dir: Thư mục lưu đồ thị loss
        img_dir: Thư mục lưu kết quả ảnh sinh
    """
    # Thiết lập thiết bị tính toán
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Sử dụng thiết bị: {device}")
    
    # Chuyển các mô hình sang thiết bị
    generator.to(device)
    discriminator.to(device)
    
    # Khởi tạo các optimizer
    optimizer_G = optim.Adam(generator.parameters(), lr=lr_g, betas=(beta1, beta2))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(beta1, beta2))
    
    # Tạo thư mục lưu checkpoint và kết quả nếu chưa tồn tại
    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(result_dir, exist_ok=True)
    
    # Khởi tạo tensors cho label real/fake
    real_label = 1.0
    fake_label = 0.0
    
    # Lists để lưu lịch sử loss
    loss_history = {
        'D_total': [], 'D_real': [], 'D_fake': [], 'D_gender': [], 'D_age': [],
        'G_total': [], 'G_adv': [], 'G_gender': [], 'G_age': []
    }
    
    # Training loop
    total_start_time = time.time()
    
    for epoch in range(1, num_epochs + 1):
        epoch_start_time = time.time()
        
        # Khởi tạo các biến tích lũy loss
        running_loss_D_total = 0.0
        running_loss_D_real = 0.0
        running_loss_D_fake = 0.0
        running_loss_D_gender = 0.0
        running_loss_D_age = 0.0
        running_loss_G_total = 0.0
        running_loss_G_adv = 0.0
        running_loss_G_gender = 0.0
        running_loss_G_age = 0.0
        
        # Lặp qua các batch dữ liệu
        with tqdm(dataloader, desc=f"Epoch {epoch}/{num_epochs}") as pbar:
            for i, batch in enumerate(pbar):
                # Di chuyển dữ liệu sang thiết bị
                real_imgs = batch['real_img'].to(device)
                real_conditions = batch['real_condition'].to(device)
                cf_conditions = batch['cf_condition'].to(device)
                
                # Lấy thông tin điều kiện gốc (không chuẩn hóa) để ghi log
                real_raw_conditions = batch['real_raw_condition'].to(device)
                cf_raw_conditions = batch['cf_raw_condition'].to(device)
                
                batch_size = real_imgs.size(0)
                
                # Chuẩn bị labels
                real_targets = torch.full((batch_size, 1), real_label, device=device)
                fake_targets = torch.full((batch_size, 1), fake_label, device=device)
                
                # Trích xuất thông tin giới tính và tuổi
                real_gender = real_conditions[:, 1].view(-1, 1)
                real_age = real_raw_conditions[:, 0].view(-1, 1)
                cf_gender = cf_conditions[:, 1].view(-1, 1)
                cf_age = cf_raw_conditions[:, 0].view(-1, 1)
                
                # -------------------------------
                # Huấn luyện Discriminator
                # -------------------------------
                optimizer_D.zero_grad()
                
                # Tính loss cho ảnh thật
                real_adv_preds, real_gender_preds, real_age_preds = discriminator(real_imgs)
                
                loss_D_real_adv = adversarial_loss(real_adv_preds, real_targets)
                loss_D_real_gender = gender_loss(real_gender_preds, real_gender)
                loss_D_real_age = age_loss(real_age_preds, real_age)
                
                loss_D_real = loss_D_real_adv + lambda_gender * loss_D_real_gender + lambda_age * loss_D_real_age
                
                # Tạo ảnh giả
                fake_imgs, _ = generator(real_imgs, cf_conditions)
                
                # Tính loss cho ảnh giả
                fake_adv_preds, fake_gender_preds, fake_age_preds = discriminator(fake_imgs.detach())
                
                loss_D_fake_adv = adversarial_loss(fake_adv_preds, fake_targets)
                loss_D_fake_gender = gender_loss(fake_gender_preds, cf_gender)
                loss_D_fake_age = age_loss(fake_age_preds, cf_age)
                
                loss_D_fake = loss_D_fake_adv + lambda_gender * loss_D_fake_gender + lambda_age * loss_D_fake_age
                
                # Tổng hợp loss cho Discriminator
                loss_D = (loss_D_real + loss_D_fake) / 2
                
                # Backward và optimize
                loss_D.backward()
                optimizer_D.step()
                
                # -------------------------------
                # Huấn luyện Generator
                # -------------------------------
                optimizer_G.zero_grad()
                
                # Sinh ảnh giả một lần nữa để tính toán loss G
                fake_imgs, _ = generator(real_imgs, cf_conditions)
                
                # Phân loại ảnh giả bằng Discriminator
                fake_adv_preds, fake_gender_preds, fake_age_preds = discriminator(fake_imgs)
                
                # Tính các loss cho Generator
                loss_G_adv = adversarial_loss(fake_adv_preds, real_targets)  # Đánh lừa D nghĩ rằng ảnh giả là thật
                loss_G_gender = gender_loss(fake_gender_preds, cf_gender)    # Khớp với giới tính mục tiêu
                loss_G_age = age_loss(fake_age_preds, cf_age)                # Khớp với tuổi mục tiêu
                
                # Tổng hợp loss cho Generator
                loss_G = lambda_adv * loss_G_adv + lambda_gender * loss_G_gender + lambda_age * loss_G_age
                
                # Backward và optimize
                loss_G.backward()
                optimizer_G.step()
                
                # -------------------------------
                # Cập nhật các loss tích lũy
                # -------------------------------
                running_loss_D_total += loss_D.item()
                running_loss_D_real += loss_D_real_adv.item()
                running_loss_D_fake += loss_D_fake_adv.item()
                running_loss_D_gender += (loss_D_real_gender.item() + loss_D_fake_gender.item()) / 2
                running_loss_D_age += (loss_D_real_age.item() + loss_D_fake_age.item()) / 2
                
                running_loss_G_total += loss_G.item()
                running_loss_G_adv += loss_G_adv.item()
                running_loss_G_gender += loss_G_gender.item()
                running_loss_G_age += loss_G_age.item()
                
                # Cập nhật thanh tiến độ
                pbar.set_postfix({
                    'D_loss': f"{loss_D.item():.4f}",
                    'G_loss': f"{loss_G.item():.4f}"
                })
                
                # Trực quan hóa kết quả cho batch cuối mỗi epoch
                if i == len(dataloader) - 1:
                    visualize_results(real_imgs, fake_imgs, 
                                     real_raw_conditions, cf_raw_conditions, 
                                     epoch, img_dir)
        
        # Tính loss trung bình cho epoch
        avg_loss_D_total = running_loss_D_total / len(dataloader)
        avg_loss_D_real = running_loss_D_real / len(dataloader)
        avg_loss_D_fake = running_loss_D_fake / len(dataloader)
        avg_loss_D_gender = running_loss_D_gender / len(dataloader)
        avg_loss_D_age = running_loss_D_age / len(dataloader)
        
        avg_loss_G_total = running_loss_G_total / len(dataloader)
        avg_loss_G_adv = running_loss_G_adv / len(dataloader)
        avg_loss_G_gender = running_loss_G_gender / len(dataloader)
        avg_loss_G_age = running_loss_G_age / len(dataloader)
        
        # Lưu lịch sử loss
        loss_history['D_total'].append(avg_loss_D_total)
        loss_history['D_real'].append(avg_loss_D_real)
        loss_history['D_fake'].append(avg_loss_D_fake)
        loss_history['D_gender'].append(avg_loss_D_gender)
        loss_history['D_age'].append(avg_loss_D_age)
        
        loss_history['G_total'].append(avg_loss_G_total)
        loss_history['G_adv'].append(avg_loss_G_adv)
        loss_history['G_gender'].append(avg_loss_G_gender)
        loss_history['G_age'].append(avg_loss_G_age)
        
        # In thông tin loss cho epoch
        epoch_time = time.time() - epoch_start_time
        print(f"Epoch [{epoch}/{num_epochs}] - {epoch_time:.2f}s")
        print(f"D_total: {avg_loss_D_total:.4f}, D_real: {avg_loss_D_real:.4f}, D_fake: {avg_loss_D_fake:.4f}, "
              f"D_gender: {avg_loss_D_gender:.4f}, D_age: {avg_loss_D_age:.4f}")
        print(f"G_total: {avg_loss_G_total:.4f}, G_adv: {avg_loss_G_adv:.4f}, "
              f"G_gender: {avg_loss_G_gender:.4f}, G_age: {avg_loss_G_age:.4f}")
        
        # Lưu checkpoint sau mỗi 'save_freq' epoch
        if epoch % save_freq == 0 or epoch == num_epochs:
            save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, 
                           loss_history, epoch, checkpoint_dir)
            
            # Vẽ đồ thị loss
            plot_losses(loss_history, epoch, result_dir)
    
    total_time = time.time() - total_start_time
    print(f"Đã hoàn thành training sau {total_time:.2f} giây")
    
    # Vẽ đồ thị loss cuối cùng
    plot_losses(loss_history, num_epochs, result_dir)
    
    return loss_history

def save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, 
                   loss_history, epoch, checkpoint_dir):
    """Lưu trạng thái của mô hình và optimizer"""
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    
    checkpoint = {
        'epoch': epoch,
        'generator': generator.state_dict(),
        'discriminator': discriminator.state_dict(),
        'optimizer_G': optimizer_G.state_dict(),
        'optimizer_D': optimizer_D.state_dict(),
        'loss_history': loss_history
    }
    
    torch.save(checkpoint, checkpoint_path)
    print(f"Đã lưu checkpoint tại epoch {epoch} vào: {checkpoint_path}\n")

def visualize_results(real_imgs, fake_imgs, real_conditions, cf_conditions, epoch, img_dir):
    """
    Trực quan hóa ảnh thật và ảnh sinh, tách thành 3 slice riêng biệt (axial, sagittal, coronal)
    
    Args:
        real_imgs: Tensor ảnh thật [batch_size, 3, H, W]
        fake_imgs: Tensor ảnh sinh [batch_size, 3, H, W]
        real_conditions: Thông tin điều kiện gốc
        cf_conditions: Thông tin điều kiện mục tiêu
        epoch: Epoch hiện tại
        img_dir: Thư mục lưu kết quả
    """
    # Chuyển về CPU
    real_imgs = real_imgs.detach().cpu()
    fake_imgs = fake_imgs.detach().cpu()
    
    # Giới hạn số lượng ảnh hiển thị
    num_samples = min(4, real_imgs.size(0))
    
    # Tên của các slice
    slice_names = ["Axial", "Sagittal", "Coronal"]
    
    # Tạo lưới ảnh: 2 hàng (real/fake) x num_samples mẫu x 3 slice mỗi mẫu
    fig, axes = plt.subplots(2 * num_samples, 3, figsize=(12, 4 * num_samples))
    
    for i in range(num_samples):
        # Lấy thông tin tuổi và giới tính
        real_age = real_conditions[i, 0].item()
        real_gender = "Nam" if real_conditions[i, 1].item() < 0.5 else "Nữ"
        cf_age = cf_conditions[i, 0].item()
        cf_gender = "Nam" if cf_conditions[i, 1].item() < 0.5 else "Nữ"
        
        # Xử lý từng slice
        for j in range(3):
            # Lấy từng slice từ ảnh 3 kênh
            real_slice = real_imgs[i, j].numpy()
            fake_slice = fake_imgs[i, j].numpy()
            
            # Chuẩn hóa các giá trị pixel về khoảng [0, 1]
            real_slice = (real_slice - real_slice.min()) / (real_slice.max() - real_slice.min() + 1e-8)
            fake_slice = (fake_slice - fake_slice.min()) / (fake_slice.max() - fake_slice.min() + 1e-8)
            
            # Hiển thị ảnh
            row_real = i * 2
            row_fake = i * 2 + 1
            
            # Hiển thị slice thật
            axes[row_real, j].imshow(real_slice, cmap='gray')
            if j == 0:  # Chỉ hiển thị thông tin tuổi/giới tính ở slice đầu tiên
                axes[row_real, j].set_title(f"{slice_names[j]} - Thật: {real_age} tuổi, {real_gender}")
            else:
                axes[row_real, j].set_title(f"{slice_names[j]}")
            axes[row_real, j].axis('off')
            
            # Hiển thị slice giả
            axes[row_fake, j].imshow(fake_slice, cmap='gray')
            if j == 0:
                axes[row_fake, j].set_title(f"{slice_names[j]} - Giả: {cf_age} tuổi, {cf_gender}")
            else:
                axes[row_fake, j].set_title(f"{slice_names[j]}")
            axes[row_fake, j].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(result_dir, f"result_epoch_{epoch}.png"), dpi=300)
    plt.close()
    
    # Tạo thêm một hình ảnh chi tiết hơn cho mẫu đầu tiên
    if num_samples > 0:
        create_detailed_visualization(
            real_imgs[0], fake_imgs[0],
            real_age, real_gender, cf_age, cf_gender,
            epoch, result_dir
        )

def create_detailed_visualization(real_img, fake_img, real_age, real_gender, cf_age, cf_gender, epoch, result_dir):
    """
    Tạo một trực quan hóa chi tiết hơn cho một mẫu để phân tích sâu hơn
    """
    slice_names = ["Axial", "Sagittal", "Coronal"]
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Hiển thị từng slice thật
    for j in range(3):
        real_slice = real_img[j].numpy()
        real_slice = (real_slice - real_slice.min()) / (real_slice.max() - real_slice.min() + 1e-8)
        axes[0, j].imshow(real_slice, cmap='gray')
        axes[0, j].set_title(f"Thật - {slice_names[j]}")
        axes[0, j].axis('off')
    
    # Hiển thị từng slice giả
    for j in range(3):
        fake_slice = fake_img[j].numpy()
        fake_slice = (fake_slice - fake_slice.min()) / (fake_slice.max() - fake_slice.min() + 1e-8)
        axes[1, j].imshow(fake_slice, cmap='gray')
        axes[1, j].set_title(f"Giả - {slice_names[j]}")
        axes[1, j].axis('off')
    
    # Hiển thị sự khác biệt giữa thật và giả (cho slice đầu tiên)
    real_slice_0 = real_img[0].numpy()
    real_slice_0 = (real_slice_0 - real_slice_0.min()) / (real_slice_0.max() - real_slice_0.min() + 1e-8)
    fake_slice_0 = fake_img[0].numpy()
    fake_slice_0 = (fake_slice_0 - fake_slice_0.min()) / (fake_slice_0.max() - fake_slice_0.min() + 1e-8)
    
    diff = np.abs(real_slice_0 - fake_slice_0)
    
    axes[0, 3].imshow(diff, cmap='hot')
    axes[0, 3].set_title("Sự khác biệt (Axial)")
    axes[0, 3].axis('off')
    
    # Hiển thị thông tin mẫu
    axes[1, 3].axis('off')
    info_text = (
        f"Thông tin mẫu:\n\n"
        f"Thật: {real_age} tuổi, {real_gender}\n"
        f"Giả: {cf_age} tuổi, {cf_gender}\n\n"
        f"Thay đổi:\n"
        f"Tuổi: {real_age} → {cf_age}\n"
        f"Giới tính: {real_gender} → {cf_gender}"
    )
    axes[1, 3].text(0.1, 0.5, info_text, fontsize=12, va='center')
    
    plt.tight_layout()
    plt.savefig(os.path.join(result_dir, f"detailed_result_epoch_{epoch}.png"), dpi=300)
    plt.close()

def plot_losses(loss_history, epoch, result_dir):
    """Vẽ đồ thị các loss theo thời gian"""
    # Tạo subplot cho loss của Discriminator
    plt.figure(figsize=(12, 10))
    
    plt.subplot(2, 1, 1)
    plt.plot(loss_history['D_total'], label='D Total')
    plt.plot(loss_history['D_real'], label='D Real')
    plt.plot(loss_history['D_fake'], label='D Fake')
    plt.plot(loss_history['D_gender'], label='D Gender')
    plt.plot(loss_history['D_age'], label='D Age')
    plt.title('Discriminator Losses')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Subplot cho loss của Generator
    plt.subplot(2, 1, 2)
    plt.plot(loss_history['G_total'], label='G Total')
    plt.plot(loss_history['G_adv'], label='G Adversarial')
    plt.plot(loss_history['G_gender'], label='G Gender')
    plt.plot(loss_history['G_age'], label='G Age')
    plt.title('Generator Losses')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(result_dir, f"losses_epoch_{epoch}.png"))
    plt.close()

In [ ]:
def main():
    """Hàm chính để khởi chạy quá trình training"""
    # Thiết lập các thông số
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size = 8
    num_workers = 4
    lr_g = 2e-4
    lr_d = 2e-4
    beta1 = 0.5
    beta2 = 0.999
    num_epochs = 100
    save_freq = 5
    checkpoint_dir = 'GAN/checkpoints'
    result_dir = 'GAN/results'
    img_dir = 'GAN/images'
    
    # Thiết lập các trọng số loss
    lambda_adv = 1.0
    lambda_gender = 0.8
    lambda_age = 0.5
    
    # Tạo dataset và dataloader
    transform = None  # Nếu cần transform thêm
    dataset = BrainMRIDataset(
        data_dir='data',
        participants_file='data/participants_1.xlsx',
        transform=transform
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )
    
    # Khởi tạo các mô hình
    generator = MultiSliceGenerator()
    discriminator = Discriminator(in_channels=3)
    
    # Khởi tạo trọng số
    generator.apply(weights_init_normal)
    discriminator.apply(weights_init_normal)
    
    # Chuyển các mô hình sang thiết bị
    generator.to(device)
    discriminator.to(device)
    
    # Khởi chạy quá trình training
    loss_history = train_gan(
        generator=generator,
        discriminator=discriminator,
        dataloader=dataloader,
        num_epochs=num_epochs,
        lr_g=lr_g,
        lr_d=lr_d,
        beta1=beta1,
        beta2=beta2,
        lambda_adv=lambda_adv,
        lambda_gender=lambda_gender,
        lambda_age=lambda_age,
        checkpoint_dir=checkpoint_dir,
        save_freq=save_freq,
        device=device,
        result_dir=result_dir,
        img_dir=img_dir
    )
    
    return loss_history

if __name__ == "__main__":
    main()

Tìm thấy 4924 mẫu có dữ liệu MRI hợp lệ
Sử dụng thiết bị: cuda


Epoch 1/100: 100%|██████████| 616/616 [02:40<00:00,  3.85it/s, D_loss=5.6082, G_loss=4.6133]  


Epoch [1/100] - 160.09s
D_total: 14.1175, D_real: 0.6216, D_fake: 0.6209, D_gender: 0.6789, D_age: 25.9063
G_total: 13.8036, G_adv: 0.8323, G_gender: 0.6794, G_age: 24.8556


Epoch 2/100: 100%|██████████| 616/616 [01:40<00:00,  6.14it/s, D_loss=4.5894, G_loss=5.7688]  


Epoch [2/100] - 100.26s
D_total: 6.0156, D_real: 0.4192, D_fake: 0.4274, D_gender: 0.6817, D_age: 10.0939
G_total: 5.6646, G_adv: 1.4410, G_gender: 0.6820, G_age: 7.3560


...

Epoch 62/100: 100%|██████████| 616/616 [01:55<00:00,  5.32it/s, D_loss=5.3271, G_loss=4.6901] 


Epoch [62/100] - 115.87s
D_total: 4.1881, D_real: 0.3295, D_fake: 0.3603, D_gender: 0.1937, D_age: 7.3765
G_total: 5.6100, G_adv: 2.2800, G_gender: 0.1108, G_age: 6.4828


Epoch 63/100: 100%|██████████| 616/616 [01:45<00:00,  5.82it/s, D_loss=3.9950, G_loss=9.2828] 


Epoch [63/100] - 105.91s
D_total: 4.1698, D_real: 0.3398, D_fake: 0.3719, D_gender: 0.1963, D_age: 7.3137
G_total: 5.5413, G_adv: 2.2443, G_gender: 0.1131, G_age: 6.4130


Epoch 64/100:  51%|█████     | 314/616 [00:54<00:51,  5.89it/s, D_loss=2.9672, G_loss=6.7778]

In [ ]:
def generate_brain_images(generator, age, gender, latent_dim=100, num_samples=5):
    """
    Tạo ảnh não dựa trên tuổi và giới tính
    
    Args:
        generator: Generator đã huấn luyện
        age: Tuổi (số thực)
        gender: Giới tính ('m' hoặc 'f')
        latent_dim: Kích thước latent vector
        num_samples: Số lượng mẫu cần tạo
    """
    generator.eval()
    with torch.no_grad():
        # Tạo nhiễu ngẫu nhiên
        z = torch.randn(num_samples, latent_dim, device=device)
        
        # Chuẩn bị điều kiện
        norm_age = age / 100.0  # Chuẩn hóa tuổi
        gender_val = 1.0 if gender == 'm' else 0.0
        condition = torch.tensor([[norm_age, gender_val]]).float().to(device)
        condition = condition.repeat(num_samples, 1)
        
        # Tạo ảnh
        generated_images = generator(z, condition)
        
        # Hiển thị kết quả
        fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
        
        for i in range(num_samples):
            for j, slice_name in enumerate(['Axial', 'Coronal', 'Sagittal']):
                if num_samples > 1:
                    ax = axes[i, j]
                else:
                    ax = axes[j]
                
                # Chuyển từ [-1, 1] về [0, 1] để hiển thị
                img = (generated_images[i, j].cpu().numpy() + 1) / 2
                ax.imshow(img, cmap='gray')
                ax.set_title(f"{slice_name} Slice")
                ax.axis('off')
        
        gender_str = "Nam" if gender == 'm' else "Nữ"
        plt.suptitle(f'Ảnh MRI não được tạo: Tuổi {age}, Giới tính {gender_str}')
        plt.tight_layout()
        plt.show()

# Thử nghiệm tạo ảnh
try:
    # Tải model đã huấn luyện (nếu có)
    checkpoint_path = 'model_checkpoint_epoch_99.pth'  # Đổi tên file nếu cần
    
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        generator.load_state_dict(checkpoint['generator'])
        print("Đã tải model đã huấn luyện!")
    
    # Tạo một số ảnh với các tham số khác nhau
    print("Tạo ảnh cho nam 35 tuổi:")
    generate_brain_images(generator, age=35, gender='m', num_samples=3)
    
    print("Tạo ảnh cho nữ 70 tuổi:")
    generate_brain_images(generator, age=70, gender='f', num_samples=3)
    
except Exception as e:
    print(f"Lỗi khi tạo ảnh: {e}")
    print("Bạn cần huấn luyện mô hình trước khi tạo ảnh.")

In [ ]:
def save_model(generator, discriminator, optimizer_G, optimizer_D, epoch, filename='gan_model.pth'):
    """Lưu trạng thái mô hình GAN"""
    torch.save({
        'generator': generator.state_dict(),
        'discriminator': discriminator.state_dict(),
        'optimizer_G': optimizer_G.state_dict(),
        'optimizer_D': optimizer_D.state_dict(),
        'epoch': epoch
    }, filename)
    print(f"Đã lưu mô hình vào {filename}")

def load_model(generator, discriminator, optimizer_G=None, optimizer_D=None, filename='gan_model.pth'):
    """Tải trạng thái mô hình GAN"""
    if os.path.exists(filename):
        checkpoint = torch.load(filename)
        generator.load_state_dict(checkpoint['generator'])
        discriminator.load_state_dict(checkpoint['discriminator'])
        
        if optimizer_G is not None:
            optimizer_G.load_state_dict(checkpoint['optimizer_G'])
        if optimizer_D is not None:
            optimizer_D.load_state_dict(checkpoint['optimizer_D'])
            
        epoch = checkpoint['epoch']
        print(f"Đã tải mô hình từ epoch {epoch}")
        return epoch
    else:
        print(f"Không tìm thấy file {filename}")
        return 0

# Ví dụ cách sử dụng
try:
    # Khởi tạo optimizer mới (chỉ để minh họa)
    optimizer_G = optim.Adam(generator.parameters(), lr=lr_g, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.5, 0.999))
    
    # Lưu mô hình
    save_model(generator, discriminator, optimizer_G, optimizer_D, num_epochs, 'gan_model_final.pth')
    
    # Tải mô hình (chỉ để minh họa)
    start_epoch = load_model(generator, discriminator, optimizer_G, optimizer_D, 'gan_model_final.pth')
    print(f"Tiếp tục huấn luyện từ epoch {start_epoch}")
    
except Exception as e:
    print(f"Lỗi khi lưu/tải mô hình: {e}")